# MOESCORE XACLE 2027 — Full8 end-to-end on Kaggle T4 ×2

This notebook builds independent profile `full_8`: original E1–E7 order plus the frozen M2D-CLAP audio/text LoRA CORAL Expert 8. It verifies and reuses a compatible Full7 store when available. Otherwise, it runs the existing Full9 → E5 → Whisper → E6 → Full7 feature-store stages. The Full7 Gate is never trained.

Attach the official XACLE dataset, the Full9 cache (unless you attach a validated Full7 store), and pretrained weights containing `m2d-lora-Nam.pt` plus the exact M2D `checkpoint-30.pth`. Exact-name discovery runs under `/kaggle/input`; optional path overrides are in the central configuration. Both checkpoint identities are verified before any E5 or Whisper work.

`ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP=True` is an explicit research exception. Validation can be optimistically biased. `SKIP_SCORE_REVIEW_EXPERIMENTAL=True` skips human review without claiming or fabricating it; the train-only Score Audit remains on. Test labels are opened only after label-free predictions are frozen.


## 00 — Central configuration

Set paths or switches here. Leave both Expert 8 checkpoint overrides as `None` when exactly one matching artifact is attached in Kaggle Inputs. `Save & Run All` performs the stages in order without a Human Score Review pause.


In [ ]:
from pathlib import Path
import os, sys, json, csv, subprocess, shutil, hashlib, tarfile, time

REPO_URL = "https://github.com/vietquang8071/MOESCORE.git"
SOURCE_COMMIT = "ffcb969951731dccca792ee86b6bdadb088b2f3c"
REPO_ROOT = Path("/kaggle/working/MOESCORE")
SKIP_SCORE_REVIEW_EXPERIMENTAL = True
RUN_TEST_METRICS = True
DATA_ROOT = Path("/kaggle/input/datasets/qnv8071/xacle-2027-data/XACLE_Challenge_2027_development_dataset")
FULL9_CACHE_INPUT = Path("/kaggle/input/datasets/qnv8071/moescore-full9-feature-cache")
# Set this to an existing train/validation/test store root if you know the exact path.
FULL9_STORE_OVERRIDE = None


WORK_ROOT = Path("/kaggle/working/moescore_full8")
CONFIG_ROOT = WORK_ROOT / "configs"
LOG_ROOT = WORK_ROOT / "logs"
WHISPER_ROOT = WORK_ROOT / "whisper_cache"
E5_ROOT = WORK_ROOT / "e5"
E6_ROOT = WORK_ROOT / "e6"
FULL7_ROOT = WORK_ROOT / "full7_store"
E8_CACHE_ROOT = WORK_ROOT / "expert8_cache"
FULL8_ROOT = WORK_ROOT / "full8_store"
GATE_ROOT_BASE = WORK_ROOT / "full8_gate"
GATE_ROOT = GATE_ROOT_BASE
METADATA_ROOT = WORK_ROOT / "whisper_metadata"
HF_E8_CACHE_ROOT = WORK_ROOT / "hf_e8_cache"
FULL7_STORE_OVERRIDE = None
E8_LORA_CHECKPOINT_OVERRIDE = None  # Exact required filename: m2d-lora-Nam.pt
E8_BASE_CHECKPOINT_OVERRIDE = None  # Compatible M2D artifact SHA is verified.

# Whisper weights: use a complete local snapshot if already attached, or download once.
WHISPER_SNAPSHOT_INPUT = None  # e.g. '/kaggle/input/.../whisper-large-v2-snapshot'
WHISPER_CACHE_DIR = WORK_ROOT / "hf_whisper" / "hub"
ALLOW_WHISPER_DOWNLOAD = True  # Only if Internet is enabled and the pretrained model is approved.
WHISPER_BATCH_SIZE = 1          # FROZEN: affects decoder padding & mean-pooling contract.
WHISPER_NUM_WORKERS = 0

INSTALL_MISSING_DEPENDENCIES = True
ALLOW_E8_BERT_DOWNLOAD = True
E8_BATCH_SIZE = 1
E8_SHARD_SIZE = 8
RUN_E5_TRAIN = True
RUN_WHISPER_SMOKE = True
RUN_WHISPER_EXTRACT = True
RUN_E6_TRAIN = True
RUN_FULL7_BUILD = True
RUN_E8_EXTRACTION = True
RUN_FULL8_BUILD = True
RUN_SCORE_AUDIT = True
RUN_GATE_TRAIN = True
RUN_VALIDATION = True
RUN_TEST_INFERENCE = True
ALLOW_POST_INFERENCE_TEST_LABEL_ACCESS = True
ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP = True

# Keep False: this is not a declaration that a human reviewed the scores.
HUMAN_SCORE_REVIEW_CONFIRMED = False
REVIEWER_NAME = ""             # Fill with a real reviewer's name, not a placeholder.
REVIEW_EVIDENCE = ""           # Location/description of the actual reviewed evidence.

# Competition: E6 selects a checkpoint by validation MSE; gate uses validation SRCC.
# Research metrics are provisional until organizer approvals/policy are confirmed.
RUN_PURPOSE = "research_only"  # Not a declaration of challenge submission compliance.
# Experimental overlap does not remove leakage or make validation independent.

E5_DEVICE = "0"
E6_DEVICE = "0"
GATE_DEVICE = "cuda:0"
SHARD_SIZE = 32
EXPECTED_COUNTS = {"train": 7500, "validation": 3000, "test": 3000}

# Never upload protected competition data to a public dataset.
CREATE_LOCAL_BACKUP = True
CREATE_PRE_GATE_BACKUP = True
UPLOAD_PRIVATE_PRE_GATE_BACKUP = False
UPLOAD_PRIVATE_BACKUP = False
PRIVATE_BACKUP_HANDLE = "qnv8071/moescore-full7-intermediates"

os.environ["HF_HOME"] = str(HF_E8_CACHE_ROOT)
os.environ["HF_HUB_CACHE"] = str(HF_E8_CACHE_ROOT / "hub")
assert WHISPER_BATCH_SIZE == 1 and E8_BATCH_SIZE == 1
for p in (WORK_ROOT, CONFIG_ROOT, LOG_ROOT, WHISPER_ROOT, E5_ROOT, E6_ROOT, FULL7_ROOT, E8_CACHE_ROOT, FULL8_ROOT, GATE_ROOT_BASE, METADATA_ROOT, WHISPER_CACHE_DIR, HF_E8_CACHE_ROOT):
    p.mkdir(parents=True, exist_ok=True)
print("Full8 work root:", WORK_ROOT)


## 01 — Source, E8 checkpoint and two-T4 preflight

The exact `m2d-lora-Nam.pt` list checkpoint and compatible M2D base checkpoint are validated before E5 or Whisper extraction. Expert 8 remains independent from the existing E7 M2D-LoRA adapter.


In [ ]:
if not REPO_ROOT.is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--detach", SOURCE_COMMIT], check=True)
current_sha = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
assert current_sha == SOURCE_COMMIT, (current_sha, SOURCE_COMMIT)

required_files = [
    "scripts/build_full7_feature_store.py", "scripts/audit_full7_scores.py",
    "scripts/train_set_a_residual.py", "scripts/train_set_b_residual.py",
    "scripts/extract_whisper_features.py", "scripts/train_moe_gating.py",
    "scripts/evaluate_moe_gating.py", "scripts/infer_moe_gating.py",
    "configs/moe_full7.example.yaml", "configs/moe_full8.example.yaml",
    "configs/expert8_extraction.example.yaml", "scripts/extract_expert8_features.py",
    "scripts/build_full8_feature_store.py", "data/full8_assembler.py",
    "model/expert_adapters/expert8_m2d_clap_adapter.py",
]
missing_files = [s for s in required_files if not (REPO_ROOT / s).is_file()]
if missing_files:
    raise FileNotFoundError(f"Expected integrated Full7 source at {SOURCE_COMMIT}: {missing_files}")

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import torch

from model.expert_adapters.expert8_m2d_clap_runtime import (
    EXPECTED_BASE_M2D_SHA256, EXPECTED_LORA_SHAPES,
    load_expert8_checkpoint, validate_base_m2d_checkpoint,
)
from data.moe_expert_cache import sha256_file

def resolve_exact_input(filename, override, local_fallback=None):
    if override is not None:
        candidate = Path(override).expanduser().resolve()
        if not candidate.is_file():
            raise FileNotFoundError(f"Checkpoint override not found: {candidate}")
        return candidate
    input_root = Path("/kaggle/input")
    matches = sorted(p.resolve() for p in input_root.rglob(filename) if p.is_file()) if input_root.is_dir() else []
    if len(matches) > 1:
        raise ValueError(f"Ambiguous {filename}: found {len(matches)} exact matches: {matches}. Set its explicit override.")
    if len(matches) == 1:
        return matches[0]
    if local_fallback is not None and Path(local_fallback).is_file():
        return Path(local_fallback).resolve()
    raise FileNotFoundError(f"Required {filename} is missing under /kaggle/input and {local_fallback}; attach the checkpoint Dataset or set an override.")

E8_LORA_CHECKPOINT = resolve_exact_input(
    "m2d-lora-Nam.pt", E8_LORA_CHECKPOINT_OVERRIDE,
    REPO_ROOT / "weights" / "m2d-lora-Nam.pt",
)
E8_BASE_CHECKPOINT = resolve_exact_input(
    "checkpoint-30.pth", E8_BASE_CHECKPOINT_OVERRIDE,
    REPO_ROOT / "weights" / "pretrained" / "checkpoint-30.pth",
)
E8_LORA_SHA256 = sha256_file(E8_LORA_CHECKPOINT)
E8_BASE_SHA256 = validate_base_m2d_checkpoint(E8_BASE_CHECKPOINT)
e8_preflight = load_expert8_checkpoint(E8_LORA_CHECKPOINT)
assert len(e8_preflight["lora"]) == len(EXPECTED_LORA_SHAPES) == 24
assert tuple(e8_preflight["head"]["net.5.weight"].shape) == (10, 256)
print("E8 trained checkpoint:", E8_LORA_CHECKPOINT, E8_LORA_SHA256)
print("E8 base checkpoint:", E8_BASE_CHECKPOINT, E8_BASE_SHA256)
print("E8 checkpoint keys, ten-logit head and 24 LoRA tensors: PASS")
del e8_preflight
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"Attach the official XACLE dataset: {DATA_ROOT}")
for split in EXPECTED_COUNTS:
    if not (DATA_ROOT / "wav" / split).is_dir():
        raise FileNotFoundError(f"Missing XACLE {split} WAV directory")
    if not (DATA_ROOT / "meta_data" / f"{split}_average.csv").is_file():
        raise FileNotFoundError(f"Missing XACLE {split} metadata CSV")

assert torch.cuda.is_available() and torch.cuda.device_count() == 2, "Select GPU T4 ×2 in Kaggle settings"
print("SOURCE COMMIT", current_sha)
for gpu_idx in range(2):
    props = torch.cuda.get_device_properties(gpu_idx)
    print(f"cuda:{gpu_idx}: {props.name}, {props.total_memory / 2**30:.2f} GiB")
subprocess.run(["nvidia-smi", "-L"], check=True)


## 02 — Kaggle dependencies (reuse the repository's Python-specific installer)

Keep Kaggle's preinstalled torch/torchaudio/torchvision stack. This follows the exact dependency installation strategy used in the latest Full9 notebook. A kernel restart may be necessary if core libraries already imported are replaced. Turn `INSTALL_MISSING_DEPENDENCIES=False` only after a verified installation.

In [ ]:
if INSTALL_MISSING_DEPENDENCIES:
    from scripts.kaggle_full9_dual_gpu import dependency_plan_for_python
    requirements_file, legacy_no_deps = dependency_plan_for_python(sys.version_info)
    installer = [sys.executable, str(REPO_ROOT / "scripts/install_kaggle_dependencies.py"),
                 "--requirements", str(requirements_file),
                 "--constraints", str(WORK_ROOT / "torch_constraints.txt"),
                 "--report-json", str(LOG_ROOT / "dependencies_install.json")]
    subprocess.run(installer, cwd=REPO_ROOT, check=True)
    if legacy_no_deps:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps",
                        "msclap==1.3.4", "laion-clap==1.1.7"], check=True)
print("Dependency setup finished. If packages changed after imports, restart kernel before continuing.")


In [ ]:
from data.moe_feature_store import validate_feature_store, preprocessing_contract
from data.audio_overlap import build_train_validation_audio_overlap_diagnostic

def verify_full7_candidate(root):
    root=Path(root).expanduser().resolve(strict=True); manifests={}
    for split,expected in EXPECTED_COUNTS.items():
        manifest=validate_feature_store(root,split=split,require_labels=split!="test")
        count=sum(shard["count"] for shard in manifest["shards"])
        if (manifest.get("expert_profile"),manifest.get("feature_width"),manifest.get("score_width"),count)!=("full_7",29959,7,expected):
            raise ValueError(f"{split}: incompatible Full7 profile, dimensions or sample count")
        if manifest["labels_present"] is (split=="test"):
            raise ValueError(f"{split}: invalid Full7 label presence")
        manifests[split]=manifest
    reference=manifests["train"]
    for split,manifest in manifests.items():
        if manifest["checkpoint_provenance"] != reference["checkpoint_provenance"]:
            raise ValueError(f"{split}: expert checkpoint provenance differs across public splits")
        if preprocessing_contract(manifest) != preprocessing_contract(reference):
            raise ValueError(f"{split}: preprocessing contracts differ across public splits")
    train=manifests["train"]["assembly_identity"]["ordered_source_fingerprints"]
    val=manifests["validation"]["assembly_identity"]["ordered_source_fingerprints"]
    test=manifests["test"]["assembly_identity"]["ordered_source_fingerprints"]
    test_hashes={r["wav_sha256"] for r in train+val}.intersection(r["wav_sha256"] for r in test)
    if test_hashes: raise ValueError(f"Full7 duplicate WAV involves official test: {sorted(test_hashes)[0]}")
    diag=build_train_validation_audio_overlap_diagnostic(train,val,allow_train_validation_audio_overlap=ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP)
    if diag["overlapping_sha256_count"] and not ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP:
        from data.audio_overlap import raise_for_audio_overlap
        raise_for_audio_overlap(diag,stage="Full7 cache verification")
    return manifests,diag

if FULL7_STORE_OVERRIDE is not None:
    FULL7_ROOT=Path(FULL7_STORE_OVERRIDE).expanduser().resolve()
    full7_manifests,full7_overlap_diagnostic=verify_full7_candidate(FULL7_ROOT)
    NEED_FULL7_BUILD=False
else:
    candidates=[]
    if FULL7_ROOT.is_dir(): candidates.append(FULL7_ROOT.resolve())
    input_root=Path("/kaggle/input")
    if input_root.is_dir():
        for mp in input_root.rglob("train/manifest.json"):
            candidate=mp.parent.parent.resolve()
            if candidate not in candidates: candidates.append(candidate)
    valid=[]; invalid=[]
    for candidate in candidates:
        try: valid.append((candidate,verify_full7_candidate(candidate)))
        except Exception as exc: invalid.append((candidate,f"{type(exc).__name__}: {exc}"))
    if len(valid)>1: raise ValueError(f"Multiple compatible Full7 stores found; set FULL7_STORE_OVERRIDE: {[str(v[0]) for v in valid]}")
    if valid:
        FULL7_ROOT=valid[0][0]; full7_manifests,full7_overlap_diagnostic=valid[0][1]; NEED_FULL7_BUILD=False
        print("Verified and reusing Full7 store:",FULL7_ROOT)
        print("Train/validation overlap:",json.dumps(full7_overlap_diagnostic,sort_keys=True))
    else:
        FULL7_ROOT=WORK_ROOT/"full7_store"; NEED_FULL7_BUILD=True
        print("No compatible Full7 store found; existing Full9/E5/Whisper/E6 stages will run.")
        for candidate,error in invalid[:10]: print("Ignored incompatible Full7 candidate:",candidate,error)
print("NEED_FULL7_BUILD =",NEED_FULL7_BUILD)


## 03 — E8 BERT resource preflight

Establish the exact `google-bert/bert-base-uncased` snapshot identity before E5 or Whisper work. If absent, fetch it now when enabled; otherwise fail before expensive stages.


In [ ]:
from scripts.extract_moe_feature_store import _text_snapshot_identity
HF_E8_CACHE_ROOT.mkdir(parents=True,exist_ok=True)
os.environ["HF_HOME"]=str(HF_E8_CACHE_ROOT); os.environ["HF_HUB_CACHE"]=str(HF_E8_CACHE_ROOT/"hub")
try:
    e8_bert_identity=_text_snapshot_identity("google-bert/bert-base-uncased",{"cache_dir":str(HF_E8_CACHE_ROOT)},{})
except FileNotFoundError:
    if not ALLOW_E8_BERT_DOWNLOAD: raise FileNotFoundError("E8 BERT tokenizer/model is missing; attach its cache or enable ALLOW_E8_BERT_DOWNLOAD.")
    from huggingface_hub import snapshot_download
    print("Fetching E8 BERT resources before E5/Whisper work.")
    snapshot_download("google-bert/bert-base-uncased",cache_dir=str(HF_E8_CACHE_ROOT),local_files_only=False)
    e8_bert_identity=_text_snapshot_identity("google-bert/bert-base-uncased",{"cache_dir":str(HF_E8_CACHE_ROOT)},{})
print("E8 BERT resource identity PASS; snapshots:",sum(len(v["snapshots"]) for v in e8_bert_identity["cache_views"]))
del e8_bert_identity


## 03 — Resolve and fully verify the **existing** Full9 store

This notebook **never** reruns nine expert backbones. The Full9 Kaggle Input may contain either a direct store or the archived `feature_cache.tar` format produced by `scripts/kaggle_full9_artifacts.py`. Archive restore is hash-verified. Source manifests are mandatory because the downstream E5/E6/Core builder needs exact WAV/caption identities.

In [ ]:
if NEED_FULL7_BUILD:
    from scripts.kaggle_full9_artifacts import restore_feature_cache, validate_full9_store
    from data.moe_feature_store import validate_feature_store
    from data.set_a_cached_dataset import _read_source_manifest

    if not DATA_ROOT.is_dir():
        raise FileNotFoundError(f"Missing official XACLE input: {DATA_ROOT}")
    for split in EXPECTED_COUNTS:
        assert (DATA_ROOT / "wav" / split).is_dir(), f"Missing audio directory: {split}"

    if FULL9_STORE_OVERRIDE is not None:
        FULL9_ROOT = Path(FULL9_STORE_OVERRIDE).expanduser().resolve()
    else:
        if not FULL9_CACHE_INPUT.exists():
            raise FileNotFoundError(f"Attach your existing Full9 cache Dataset: {FULL9_CACHE_INPUT}. Available: {list(Path('/kaggle/input').iterdir())}")
        roots = []
        for candidate_manifest in FULL9_CACHE_INPUT.rglob("train/manifest.json"):
            candidate = candidate_manifest.parent.parent
            if all((candidate / split / "manifest.json").is_file() for split in EXPECTED_COUNTS):
                try:
                    body = json.loads(candidate_manifest.read_text())
                    if body.get("expert_profile") == "full_9":
                        roots.append(candidate)
                except (OSError, ValueError):
                    pass
        if len(roots) == 1:
            FULL9_ROOT = roots[0]
        elif len(roots) > 1:
            raise ValueError(f"Multiple Full9 store roots; set FULL9_STORE_OVERRIDE explicitly: {roots}")
        else:
            # The Kaggle upload format contains feature_cache.tar + upload_manifest.json.
            archives = list(FULL9_CACHE_INPUT.rglob("upload_manifest.json"))
            if len(archives) != 1:
                raise FileNotFoundError("No direct Full9 store; expected exactly one upload_manifest.json + feature_cache.tar. Set FULL9_STORE_OVERRIDE if needed.")
            destination = WORK_ROOT / "full9_restore"
            archive_dir = archives[0].parent
            if not (destination / "feature_cache").is_dir():
                restore_feature_cache(archive_dir, destination)
            roots = []
            for manifest_path in (destination / "feature_cache").rglob("train/manifest.json"):
                candidate = manifest_path.parent.parent
                if all((candidate / split / "manifest.json").is_file() for split in EXPECTED_COUNTS):
                    if json.loads(manifest_path.read_text()).get("expert_profile") == "full_9":
                        roots.append(candidate)
            if len(roots) != 1:
                raise ValueError(f"Expected one validated Full9 assembly after restore, found {roots}")
            FULL9_ROOT = roots[0]

    full9_report = validate_full9_store(FULL9_ROOT, EXPECTED_COUNTS)
    source_records = {}
    for split, expected in EXPECTED_COUNTS.items():
        manifest = validate_feature_store(FULL9_ROOT, split=split, require_labels=split != "test")
        records = _read_source_manifest(FULL9_ROOT / split / "source_manifest.json", split=split,
                                        expected_sha256=manifest["source_manifest_sha256"])
        assert len(records) == expected
        if split == "test" and any(record.get("label_raw") is not None for record in records):
            raise ValueError("Full9 test source manifest must be label-free")
        source_records[split] = records

    print(json.dumps({"full9_root": str(FULL9_ROOT), "splits": full9_report}, indent=2))
    (LOG_ROOT / "full9_verified.json").write_text(json.dumps({"full9_root": str(FULL9_ROOT), "splits": full9_report}, indent=2))


## 04 — Prepare **label-free** Whisper metadata from the signed Full9 source manifest

Do not use the official `test_average.csv` or test scores. The Whisper dataset only needs WAV basename and exact raw caption. Use a stable manifest-derived row order; no inner joins, reordering, or normalization.

In [ ]:
if NEED_FULL7_BUILD:
    for split, records in source_records.items():
        target = METADATA_ROOT / f"{split}_whisper.csv"
        with target.open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=["wav_file_name", "text"])
            writer.writeheader()
            for record in records:
                writer.writerow({"wav_file_name": record["wave_file_name"], "text": record["raw_caption"]})
        with target.open(encoding="utf-8", newline="") as handle:
            assert sum(1 for _ in csv.DictReader(handle)) == len(records)
        print(split, "metadata:", target, "rows:", len(records))


## 05 — Train or verify E5 (Set A) from Full9 only

E5 learns the residual head over Top2 M2D + MS-CLAP + MGA features. No backbones and no Whisper. It trains on all official train rows and evaluates, selects, and early-stops on all official validation rows.

In [ ]:
import yaml

def run_logged(name, cmd, *, env=None):
    target = LOG_ROOT / f"{name}.log"
    print("RUN", name, "=>", target, flush=True)
    try:
        with target.open("w", encoding="utf-8") as output:
            subprocess.run([str(v) for v in cmd], cwd=REPO_ROOT, env=env,
                           stdout=output, stderr=subprocess.STDOUT, check=True)
    except subprocess.CalledProcessError:
        print("FAILED", name, "log tail:", target.read_text(errors="replace")[-3000:])
        raise
    print("PASS", name, "tail:", target.read_text(errors="replace")[-1000:], flush=True)

def gpu_env(physical_gpu):
    environment = os.environ.copy()
    environment["CUDA_VISIBLE_DEVICES"] = str(physical_gpu)
    environment["TOKENIZERS_PARALLELISM"] = "false"
    return environment

if NEED_FULL7_BUILD:
    E5_CONFIG = CONFIG_ROOT / "e5_full7.yaml"
    e5_cfg = yaml.safe_load((REPO_ROOT / "configs/set_a_residual.example.yaml").read_text())
    e5_cfg["feature_store_root"] = str(FULL9_ROOT)
    e5_cfg["output_dir"] = str(E5_ROOT)
    e5_cfg["training"]["selection_metric"] = "official_validation_mse"
    e5_cfg["runtime"]["device"] = "cuda"
    E5_CONFIG.write_text(yaml.safe_dump(e5_cfg, sort_keys=False))
    E5_BEST = E5_ROOT / "expert5_best.pt"
    E5_FINAL = E5_ROOT / "expert5_final.pt"
    E5_LATEST = E5_ROOT / "expert5_latest.pt"
    E5_SELECTION_POLICY = "official_validation_mse_v1"
    def e5_checkpoint_policy(path):
        if not path.is_file():
            return None
        payload = torch.load(path, map_location="cpu", weights_only=False)
        return payload.get("training_state", {}).get("selection_policy")

    # Old internal-holdout outputs are retrained automatically; they are never passed to resume.
    needs_e5_training = RUN_E5_TRAIN and (
        not E5_BEST.is_file() or not E5_FINAL.is_file()
        or e5_checkpoint_policy(E5_BEST) != E5_SELECTION_POLICY
        or e5_checkpoint_policy(E5_FINAL) != E5_SELECTION_POLICY
    )
    if needs_e5_training:
        command = [sys.executable, REPO_ROOT / "scripts/train_set_a_residual.py", "--config", E5_CONFIG]
        if e5_checkpoint_policy(E5_LATEST) == E5_SELECTION_POLICY:
            command += ["--resume", E5_LATEST]
        run_logged("e5_train", command, env=gpu_env(E5_DEVICE))
    if not E5_BEST.is_file() or not E5_FINAL.is_file():
        raise FileNotFoundError("E5 training incomplete: both expert5_best.pt and expert5_final.pt are required before E6/Full7.")
    from data.set_a_cached_dataset import SetACachedDataset
    from model.expert_adapters.set_a_residual_adapter import SetAResidualAdapter
    from model.composite_experts.set_a_residual import SetAResidualHead
    from training.set_a_trainer import load_set_a_checkpoint

    e5_data = SetACachedDataset(FULL9_ROOT, split="train", require_labels=True)
    e5_validation_data = SetACachedDataset(FULL9_ROOT, split="validation", require_labels=True)
    e5_model = SetAResidualAdapter(SetAResidualHead(**e5_cfg["model"]))
    e5_checkpoint = load_set_a_checkpoint(E5_BEST, model=e5_model, expected_dataset=e5_data)
    if RUN_E5_TRAIN and e5_checkpoint["training_state"].get("selection_policy") != E5_SELECTION_POLICY:
        raise ValueError("E5 training did not produce an official-validation checkpoint.")
    print("E5 train/validation rows:", len(e5_data), len(e5_validation_data))
    print("E5 verified:", E5_BEST, "bytes:", E5_BEST.stat().st_size)
    del e5_model, e5_data, e5_validation_data, e5_checkpoint


## 06 — Resolve/download the exact Whisper-large-v2 snapshot once

Both GPU workers read the same immutable snapshot. If Internet is off, attach a complete local Whisper-large-v2 snapshot as a Kaggle Input and set `WHISPER_SNAPSHOT_INPUT`. Never switch the Whisper model variant. The `whisper_top2` runtime records checkpoint file hashes.

In [ ]:
if NEED_FULL7_BUILD:
    from huggingface_hub import snapshot_download

    if WHISPER_SNAPSHOT_INPUT is not None:
        snapshot_path = Path(WHISPER_SNAPSHOT_INPUT).resolve()
        if not (snapshot_path / "config.json").is_file():
            raise FileNotFoundError(f"Whisper local snapshot incomplete: {snapshot_path}")
        whisper_model_ref = str(snapshot_path)
    else:
        if ALLOW_WHISPER_DOWNLOAD:
            # Select only PyTorch checkpoint, config and tokenizer assets (not TF/Flax weights).
            snapshot_path = Path(snapshot_download(
                repo_id="openai/whisper-large-v2",
                cache_dir=str(WHISPER_CACHE_DIR), local_files_only=False,
                allow_patterns=["*.json", "*.txt", "*.model", "*.safetensors", "*.tiktoken"],
            ))
            from model.feature_extractors.whisper_top2 import _local_model_files
            try:
                _local_model_files(snapshot_path)
            except FileNotFoundError:
                # Some HF snapshots publish only PyTorch .bin weights.
                snapshot_path = Path(snapshot_download(
                    repo_id="openai/whisper-large-v2",
                    cache_dir=str(WHISPER_CACHE_DIR), local_files_only=False,
                    allow_patterns=["*.bin", "*.index.json"],
                ))
        else:
            snapshot_path = Path(snapshot_download(
                repo_id="openai/whisper-large-v2", cache_dir=str(WHISPER_CACHE_DIR),
                local_files_only=True,
            ))
        whisper_model_ref = "openai/whisper-large-v2"
    assert (snapshot_path / "config.json").is_file()
    WHISPER_CONFIG = CONFIG_ROOT / "whisper_full7.yaml"
    whisper_cfg = yaml.safe_load((REPO_ROOT / "configs/whisper_top2.example.yaml").read_text())
    whisper_cfg["whisper_top2"]["model_name_or_path"] = whisper_model_ref
    whisper_cfg["runtime"].update({"device": "auto", "offline": True, "cache_dir": str(WHISPER_CACHE_DIR)})
    WHISPER_CONFIG.write_text(yaml.safe_dump(whisper_cfg, sort_keys=False))
    print("Whisper snapshot:", snapshot_path)
    print("Whisper config:", WHISPER_CONFIG)


## 07 — Whisper smoke on both physical T4 GPUs

Each worker has `CUDA_VISIBLE_DEVICES` set to one physical GPU, and internally uses logical `cuda:0`; this avoids real E6 cache metadata differing between train and validation. Always use batch size **1**, including resumed runs.

In [ ]:
if NEED_FULL7_BUILD:
    def whisper_command(split, *, smoke=False):
        cmd = [sys.executable, str(REPO_ROOT / "scripts/extract_whisper_features.py"),
               "--config", str(WHISPER_CONFIG),
               "--metadata-csv", str(METADATA_ROOT / f"{split}_whisper.csv"),
               "--audio-dir", str(DATA_ROOT / "wav" / split),
               "--split", split,
               "--output-dir", str(WHISPER_ROOT),
               "--batch-size", str(WHISPER_BATCH_SIZE),
               "--num-workers", str(WHISPER_NUM_WORKERS)]
        if smoke:
            cmd += ["--smoke-test"]
        return cmd

    if RUN_WHISPER_SMOKE:
        run_logged("whisper_train_gpu0_smoke", whisper_command("train", smoke=True), env=gpu_env(0))
        run_logged("whisper_validation_gpu1_smoke", whisper_command("validation", smoke=True), env=gpu_env(1))
    print("Whisper smoke complete. Verify [1,1280] audio and text shapes in both logs.")


## 08 — Dual-T4 Whisper extraction with safe split isolation and resumable caches

GPU 0 extracts **train**. GPU 1 extracts **validation then test**, sequentially, while train runs independently. Two *subprocesses* keep GPU contexts isolated. Each split writes only to `whisper_cache/<split>` and supports resuming a validated prefix. Log files can be inspected if Kaggle stops a worker.

In [ ]:
if NEED_FULL7_BUILD:
    def start_whisper(split, physical_gpu):
        log_path = LOG_ROOT / f"whisper_{split}_gpu{physical_gpu}.log"
        stream = log_path.open("a", encoding="utf-8")
        stream.write(f"\n== START {split} on physical GPU {physical_gpu}: {time.ctime()} ==\n")
        stream.flush()
        process = subprocess.Popen([str(v) for v in whisper_command(split)],
                                   cwd=REPO_ROOT, env=gpu_env(physical_gpu),
                                   stdout=stream, stderr=subprocess.STDOUT)
        return process, stream, log_path

    if RUN_WHISPER_EXTRACT:
        train_process, train_log_stream, train_log_path = start_whisper("train", 0)
        val_process, val_log_stream, val_log_path = start_whisper("validation", 1)
        try:
            val_rc = val_process.wait()
            val_log_stream.close()
            test_rc = None
            if val_rc == 0:
                test_process, test_log_stream, test_log_path = start_whisper("test", 1)
                try:
                    test_rc = test_process.wait()
                finally:
                    test_log_stream.close()
            train_rc = train_process.wait()
        finally:
            if train_process.poll() is None:
                train_process.terminate()
                train_process.wait()
            train_log_stream.close()
            if val_process.poll() is None:
                val_process.terminate()
                val_process.wait()
            val_log_stream.close()
        statuses = {"train": train_rc, "validation": val_rc, "test": test_rc}
        print("Whisper exit codes:", statuses)
        if statuses != {"train": 0, "validation": 0, "test": 0}:
            for name in ("train", "validation", "test"):
                paths = sorted(LOG_ROOT.glob(f"whisper_{name}_gpu*.log"))
                if paths:
                    print(f"{name} log tail:\n{paths[-1].read_text(errors='replace')[-2000:]}")
            raise RuntimeError(f"Whisper extraction failed; validated prefixes remain for resume: {statuses}")
    else:
        print("Whisper extraction skipped by configuration; existing caches still require verification.")


## 09 — Validate all Whisper splits and Full9 ↔ Whisper alignment

This step reads the actual original WAV bytes and tests their hashes through `SetBCachedDataset`, verifies order and 1280+1280 dimensions, and catches GPU/preprocessing provenance divergence before E6 training. Do not patch the cache if validation fails.

In [ ]:
if NEED_FULL7_BUILD:
    from data.set_b_cached_dataset import SetBCachedDataset
    whisper_datasets = {}
    for split, expected in EXPECTED_COUNTS.items():
        dataset = SetBCachedDataset(FULL9_ROOT, WHISPER_ROOT, split=split,
                                    source_audio_dir=DATA_ROOT / "wav" / split)
        assert len(dataset) == expected
        sample = dataset[0]
        assert sample["whisper_audio"].shape == (1280,)
        assert sample["whisper_text"].shape == (1280,)
        whisper_datasets[split] = dataset
        print(f"Whisper {split} PASS: {len(dataset)} rows, sample={sample['sample_id']}")
    train_dataset = whisper_datasets["train"]
    for split in ("validation", "test"):
        current = whisper_datasets[split]
        assert train_dataset.whisper_checkpoint_identity == current.whisper_checkpoint_identity, "Whisper checkpoint differs"
        assert train_dataset.whisper_preprocessing_identity == current.whisper_preprocessing_identity, "Whisper preprocessing differs"
    print("Whisper E6 provenance and alignment PASS")
    del whisper_datasets


## 10 — Train or verify E6 (Set B) using Full9 + Whisper

E6 has a fixed 30-epoch training policy; its existing code selects the best checkpoint by validation MSE. This behavior is **provisional for challenge compliance** and must be clarified with organizers. Use the existing checkpoint loader, not a hand-built substitute.

In [ ]:
if NEED_FULL7_BUILD:
    E6_CONFIG = CONFIG_ROOT / "e6_full7.yaml"
    e6_cfg = yaml.safe_load((REPO_ROOT / "configs/set_b_residual.example.yaml").read_text())
    e6_cfg["feature_store_root"] = str(FULL9_ROOT)
    e6_cfg["whisper_cache_root"] = str(WHISPER_ROOT)
    e6_cfg["output_dir"] = str(E6_ROOT)
    e6_cfg["runtime"]["device"] = "cuda"
    e6_cfg["training"]["allow_train_validation_audio_overlap"] = ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP
    e6_cfg["source_audio_dirs"] = {s: str(DATA_ROOT / "wav" / s) for s in EXPECTED_COUNTS}
    E6_CONFIG.write_text(yaml.safe_dump(e6_cfg, sort_keys=False))
    E6_BEST = E6_ROOT / "expert6_best.pt"
    E6_LATEST = E6_ROOT / "expert6_latest.pt"
    E6_FINAL = E6_ROOT / "expert6_final.pt"
    from data.set_b_cached_dataset import SetBCachedDataset
    e6_expected_datasets = {
        split: SetBCachedDataset(FULL9_ROOT, WHISPER_ROOT, split=split)
        for split in ("train", "validation")
    }
    e6_expected_provenance = {split: dataset.provenance for split, dataset in e6_expected_datasets.items()}
    from data.audio_overlap import build_train_validation_audio_overlap_diagnostic
    e6_expected_overlap_diagnostic = build_train_validation_audio_overlap_diagnostic(
        e6_expected_datasets["train"].records, e6_expected_datasets["validation"].records,
        allow_train_validation_audio_overlap=ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP,
    )
    def e6_checkpoint_matches_experiment(path):
        if not path.is_file():
            return False
        try:
            payload = torch.load(path, map_location="cpu", weights_only=False)
        except Exception:
            return False
        settings = payload.get("training_settings")
        return (payload.get("format_version") == 2
                and isinstance(settings, dict)
                and settings.get("allow_train_validation_audio_overlap") == ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP
                and settings.get("selection_policy") == "official_validation_mse_min_v1"
                and payload.get("dataset_provenance") == e6_expected_provenance
                and payload.get("audio_overlap_diagnostic") == e6_expected_overlap_diagnostic
                and payload.get("audio_overlap_diagnostic_sha256") == e6_expected_overlap_diagnostic["diagnostic_sha256"])
    e6_complete_for_experiment = all(e6_checkpoint_matches_experiment(path) for path in (E6_BEST, E6_FINAL))
    e6_latest_can_resume = e6_checkpoint_matches_experiment(E6_LATEST)
    if e6_latest_can_resume:
        e6_latest_can_resume = torch.load(E6_LATEST, map_location="cpu", weights_only=False).get("epoch", 30) < 30
    # Legacy or strict-policy checkpoints are kept for inspection but do not satisfy this experiment.
    if RUN_E6_TRAIN and not e6_complete_for_experiment:
        if RUN_PURPOSE != "research_only":
            print("WARNING: Confirm organizer validation-driven checkpoint selection approval before challenge submission.")
        command = [sys.executable, REPO_ROOT / "scripts/train_set_b_residual.py", "--config", E6_CONFIG]
        if e6_latest_can_resume:
            command += ["--resume", E6_LATEST]
        run_logged("e6_train", command, env=gpu_env(E6_DEVICE))
    if not E6_BEST.is_file() or not E6_FINAL.is_file():
        raise FileNotFoundError("E6 training incomplete: require both expert6_best.pt and expert6_final.pt (30 epochs).")
    from model.composite_experts.set_b_residual import SetBResidualHead
    from model.expert_adapters.set_b_residual_adapter import SetBResidualAdapter
    from training.set_b_trainer import load_set_b_checkpoint
    verified_e6_model = SetBResidualAdapter(SetBResidualHead(**e6_cfg["model"]))
    load_set_b_checkpoint(E6_BEST, model=verified_e6_model,
                          expected_dataset=SetBCachedDataset(FULL9_ROOT, WHISPER_ROOT, split="train"))
    print("E6 verified:", E6_BEST, "bytes:", E6_BEST.stat().st_size)
    del verified_e6_model


## 11 — Assemble and validate verified Full7 v3 (CPU; no backbones)

Uses *exact* existing CLI: Full9 → E5/E6 frozen heads → seven score columns and **29,959 features** in E1…E7 order. It binds source WAV/Whisper/E5/E6 provenance and rejects inconsistent labels or cache fingerprints.

In [ ]:
if NEED_FULL7_BUILD:
    from data.moe_feature_store import validate_feature_store
    FULL7_BUILD_REPORT = LOG_ROOT / "full7_build.json"
    if RUN_FULL7_BUILD:
        run_logged("full7_build", [
            sys.executable, REPO_ROOT / "scripts/build_full7_feature_store.py",
            "--full9-root", FULL9_ROOT,
            "--whisper-root", WHISPER_ROOT,
            "--e5-checkpoint", E5_BEST,
            "--e6-checkpoint", E6_BEST,
            "--source-audio-root", DATA_ROOT / "wav",
            "--output-root", FULL7_ROOT,
            "--split", "train", "--split", "validation", "--split", "test",
            "--shard-size", str(SHARD_SIZE),
            "--report-json", FULL7_BUILD_REPORT,
        ] + (["--allow-train-validation-audio-overlap"]
             if ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP else []))
    for split, count in EXPECTED_COUNTS.items():
        manifest = validate_feature_store(FULL7_ROOT, split=split, require_labels=split != "test")
        actual = sum(shard["count"] for shard in manifest["shards"])
        assert (manifest["expert_profile"], manifest["feature_width"], manifest["score_width"], actual) == ("full_7", 29959, 7, count)
        assert manifest["labels_present"] is (split != "test")
        print(split, "FULL7 PASS", actual, manifest["store_identity_sha256"])


## 07 — Verify all Full7 splits for Full8 assembly


In [ ]:
full7_manifests={}
for split,expected in EXPECTED_COUNTS.items():
    manifest=validate_feature_store(FULL7_ROOT,split=split,require_labels=split!="test")
    count=sum(shard["count"] for shard in manifest["shards"])
    if (manifest.get("expert_profile"),manifest.get("feature_width"),manifest.get("score_width"),count)!=("full_7",29959,7,expected): raise ValueError(f"{split}: Full7 input contract mismatch")
    if manifest["labels_present"] is (split=="test"): raise ValueError(f"{split}: Full7 test store must remain label-free")
    full7_manifests[split]=manifest
    print(split,"Full7 verified",count,manifest["store_identity_sha256"])


## 08 — Frozen E8 extraction on one T4 with resumable shards

Batch size one avoids concurrent GPU-heavy workloads. The trained LoRA and CORAL head are loaded for inference only. Test labels are projected out before dataset construction.


In [ ]:
import yaml
E8_CONFIG=CONFIG_ROOT/"expert8_extraction.yaml"
e8_cfg=yaml.safe_load((REPO_ROOT/"configs/expert8_extraction.example.yaml").read_text())
e8_cfg["runtime"].update({"device":"cuda:0","offline":True,"cache_dir":str(HF_E8_CACHE_ROOT)})
e8_cfg["experts"]["expert8"]["base_checkpoint"]=str(E8_BASE_CHECKPOINT)
e8_cfg["experts"]["expert8"]["finetuned_checkpoint"]=str(E8_LORA_CHECKPOINT)
E8_CONFIG.write_text(yaml.safe_dump(e8_cfg,sort_keys=False))
E8_EXTRACTION_REPORT=LOG_ROOT/"expert8_extraction.json"
if RUN_E8_EXTRACTION:
    run_logged("expert8_extract",[sys.executable,REPO_ROOT/"scripts/extract_expert8_features.py","--config",E8_CONFIG,"--dataset-root",DATA_ROOT,"--output-root",E8_CACHE_ROOT,"--device","cuda:0","--batch-size",str(E8_BATCH_SIZE),"--shard-size",str(E8_SHARD_SIZE),"--report-json",E8_EXTRACTION_REPORT],env=gpu_env(0))
if not E8_EXTRACTION_REPORT.is_file(): raise FileNotFoundError("Expert8 extraction report missing; enable RUN_E8_EXTRACTION or provide its verified report/cache.")
e8_extraction=json.loads(E8_EXTRACTION_REPORT.read_text())
if e8_extraction.get("status")!="PASS" or e8_extraction.get("lora_checkpoint_sha256")!=E8_LORA_SHA256 or e8_extraction.get("base_m2d_checkpoint_sha256")!=E8_BASE_SHA256: raise ValueError("E8 extraction checkpoint provenance differs from preflight")
E8_CACHE_ID=e8_extraction["expert_cache_identity"]["cache_id"]
print("E8 cache identity:",E8_CACHE_ID,"LoRA SHA-256:",E8_LORA_SHA256)


## 09 — Assemble Full8 and run train-only Score Audit


In [ ]:
FULL8_BUILD_REPORT=LOG_ROOT/"full8_build.json"
if RUN_FULL8_BUILD:
    command=[sys.executable,REPO_ROOT/"scripts/build_full8_feature_store.py","--full7-root",FULL7_ROOT,"--expert8-cache-root",E8_CACHE_ROOT,"--expert8-cache-id",E8_CACHE_ID,"--output-root",FULL8_ROOT,"--split","train","--split","validation","--split","test","--report-json",FULL8_BUILD_REPORT]
    if ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP: command.append("--allow-train-validation-audio-overlap")
    run_logged("full8_build",command)
full8_manifests={}
for split,expected in EXPECTED_COUNTS.items():
    manifest=validate_feature_store(FULL8_ROOT,split=split,require_labels=split!="test")
    count=sum(shard["count"] for shard in manifest["shards"])
    if (manifest.get("expert_profile"),manifest.get("feature_width"),manifest.get("score_width"),count)!=("full_8",33032,8,expected): raise ValueError(f"{split}: Full8 profile/dimension/count mismatch")
    if manifest["labels_present"] is (split=="test"): raise ValueError(f"{split}: Full8 test store must not contain labels")
    full8_manifests[split]=manifest; print(split,"Full8 verified",count,manifest["store_identity_sha256"])
if RUN_SCORE_AUDIT:
    FULL8_AUDIT_PATH=LOG_ROOT/"full8_score_audit.json"
    run_logged("full8_train_only_score_audit",[sys.executable,REPO_ROOT/"scripts/audit_full7_scores.py","--store-root",FULL8_ROOT,"--output-json",FULL8_AUDIT_PATH])
    full8_audit=json.loads(FULL8_AUDIT_PATH.read_text())
    if full8_audit.get("status")!="PASS" or set(full8_audit["diagnostics"]["experts"])!=set(full8_manifests["train"]["expert_order"]): raise ValueError("Full8 train-only Score Audit failed or omitted an expert")


## 10 — Full8 Gate configuration and compatibility-safe resume selection

A completed run is reused only when manifest, latest and best checkpoints match current source, configuration, train/validation stores, all eight expert identities, review policy and overlap diagnostic. Incomplete runs resume only when both checkpoints are compatible. Unsafe artifacts remain preserved in their old directory; fresh runs have no `--resume`.


In [ ]:
from training.config import load_moe_training_config
from training.checkpoint_manager import FULL8_CHECKPOINT_FORMAT_VERSION,FULL8_GATE_ARCHITECTURE,full8_source_code_sha256,_fingerprint,MoeCheckpointManager
from training.full8_resume_policy import choose_full8_gate_run,gate_resume_args
from data.moe_training_dataset import MoeTrainingDataset
from scripts.train_moe_gating import _checkpoint_config,_store_provenance,_real_score_review
from model.moe_gating_core import GatingNetwork
from training.moe_trainer import MoeGateTrainer
GATE_CONFIG=CONFIG_ROOT/"moe_full8.yaml"
GATE_TEMPLATE=yaml.safe_load((REPO_ROOT/"configs/moe_full8.example.yaml").read_text())
GATE_TEMPLATE["data"].update({"feature_store_root":str(FULL8_ROOT),"profile":"full_8","allow_train_validation_audio_overlap":ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP})
GATE_TEMPLATE["runtime"]["device"]=GATE_DEVICE
if SCORE_REVIEW_FILE_OVERRIDE is not None: GATE_TEMPLATE["data"]["score_review_file"]=str(Path(SCORE_REVIEW_FILE_OVERRIDE).expanduser().resolve())
GATE_TEMPLATE["output_dir"]=str(GATE_ROOT)
def write_gate_config(): GATE_CONFIG.write_text(yaml.safe_dump(GATE_TEMPLATE,sort_keys=False))
train_gate_ds=MoeTrainingDataset(str(FULL8_ROOT),split="train",require_labels=True)
validation_gate_ds=MoeTrainingDataset(str(FULL8_ROOT),split="validation",require_labels=True)
expert_checkpoint_provenance,GATE_STORE_IDENTITY=_store_provenance(train_gate_ds,validation_gate_ds,allow_train_validation_audio_overlap=ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP)
synthetic_provenance=bool(train_gate_ds.manifest["assembly_identity"].get("synthetic_provenance"))
REVIEW_POLICY="SYNTHETIC_NOT_APPLICABLE" if synthetic_provenance else ("EXPERIMENTAL_SKIPPED" if SKIP_SCORE_REVIEW_EXPERIMENTAL else "HUMAN_REVIEWED")
GATE_STORE_IDENTITY["score_review_policy"]=REVIEW_POLICY
if REVIEW_POLICY == "HUMAN_REVIEWED":
    if not RUN_SCORE_AUDIT or "full8_audit" not in globals():
        raise ValueError("Strict Full8 mode requires the completed train-only Score Audit")
    GATE_STORE_IDENTITY["score_review_sha256"] = _real_score_review(
        SCORE_REVIEW_FILE_OVERRIDE, train_gate_ds.manifest, full8_audit,
    )
if not SKIP_SCORE_REVIEW_EXPERIMENTAL and not synthetic_provenance and SCORE_REVIEW_FILE_OVERRIDE is None: raise ValueError("Strict mode requires a genuine review file in SCORE_REVIEW_FILE_OVERRIDE")
write_gate_config(); loaded_config=load_moe_training_config(GATE_CONFIG); loaded_config["data"]["score_review_policy"]=REVIEW_POLICY
EXPECTED_RUN_CONFIG=_checkpoint_config(loaded_config)
EXPECTED_CONFIG_SHA256=hashlib.sha256(json.dumps(EXPECTED_RUN_CONFIG,sort_keys=True,separators=(",",":"),allow_nan=False).encode()).hexdigest()
EXPECTED_SOURCE_CODE_SHA256=full8_source_code_sha256(); EXPECTED_OVERLAP_DIAGNOSTIC=GATE_STORE_IDENTITY["audio_overlap_diagnostic"]

def compatible_gate_checkpoint(path):
    path=Path(path)
    if not path.is_file(): return False
    try:
        probe=GatingNetwork(33032,8,profile="full_8")
        trainer_probe=MoeGateTrainer(
            probe,EXPECTED_RUN_CONFIG["training"]["loss"],
            learning_rate=EXPECTED_RUN_CONFIG["training"]["learning_rate"],
            weight_decay=EXPECTED_RUN_CONFIG["training"]["weight_decay"],
            diversity_weight=EXPECTED_RUN_CONFIG["training"]["diversity_weight"],
        )
        payload=MoeCheckpointManager().load(
            path,gating_network=probe,optimizer=trainer_probe.optimizer,
            scheduler=trainer_probe.scheduler,expected_config=EXPECTED_RUN_CONFIG,
            expected_feature_store_identity=GATE_STORE_IDENTITY,
            expected_expert_checkpoint_provenance=expert_checkpoint_provenance,
            restore_rng=False,
        )
        del trainer_probe,probe
        return (payload.get("format_version")==FULL8_CHECKPOINT_FORMAT_VERSION
            and payload.get("gate_architecture")==FULL8_GATE_ARCHITECTURE
            and payload.get("source_code_sha256")==EXPECTED_SOURCE_CODE_SHA256
            and payload.get("config")==EXPECTED_RUN_CONFIG
            and payload.get("config_sha256")==_fingerprint(payload.get("config"),"config")==EXPECTED_CONFIG_SHA256
            and payload.get("feature_store_identity")==GATE_STORE_IDENTITY
            and payload.get("feature_store_identity_sha256")==_fingerprint(payload.get("feature_store_identity"),"feature_store_identity")
            and payload.get("expert_checkpoint_provenance")==expert_checkpoint_provenance
            and payload.get("expert_checkpoint_provenance_sha256")==_fingerprint(payload.get("expert_checkpoint_provenance"),"expert_checkpoint_provenance")
            and payload.get("feature_store_identity",{}).get("score_review_policy")==REVIEW_POLICY
            and payload.get("feature_store_identity",{}).get("audio_overlap_diagnostic")==EXPECTED_OVERLAP_DIAGNOSTIC
            and payload.get("selection_policy")=="official_validation_srcc_strict_v1_provisional")
    except Exception as exc:
        print("Gate checkpoint incompatible:",path,f"{type(exc).__name__}: {exc}"); return False

def file_sha(path): return sha256_file(path) if Path(path).is_file() else None
def completed_run_compatible(path,latest_ok,best_ok):
    if not latest_ok or not best_ok or not Path(path).is_file(): return False
    try:
        body=json.loads(Path(path).read_text())
        return (body.get("expert_profile")=="full_8" and body.get("source_code_sha256")==EXPECTED_SOURCE_CODE_SHA256
            and body.get("config_sha256")==EXPECTED_CONFIG_SHA256
            and body.get("train_store_identity_sha256")==train_gate_ds.identity
            and body.get("validation_store_identity_sha256")==validation_gate_ds.identity
            and body.get("expert_checkpoint_provenance")==expert_checkpoint_provenance
            and body.get("score_review_policy")==REVIEW_POLICY
            and body.get("audio_overlap_diagnostic")==EXPECTED_OVERLAP_DIAGNOSTIC
            and body.get("allow_train_validation_audio_overlap")==ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP
            and body.get("gate_checkpoint_sha256")==file_sha(GATE_ROOT/"best_gate.pt")
            and body.get("latest_gate_checkpoint_sha256")==file_sha(GATE_ROOT/"latest_gate.pt"))
    except Exception as exc: print("Gate manifest incompatible:",path,f"{type(exc).__name__}: {exc}"); return False

latest,best,completed=GATE_ROOT/"latest_gate.pt",GATE_ROOT/"best_gate.pt",GATE_ROOT/"run_manifest.json"
latest_compatible=compatible_gate_checkpoint(latest); best_compatible=compatible_gate_checkpoint(best)
completed_compatible=completed_run_compatible(completed,latest_compatible,best_compatible)
has_artifacts=latest.exists() or best.exists() or completed.exists()
safe_resume_pair=latest_compatible and best_compatible and not completed.exists()
isolated_root=None
if has_artifacts and not completed_compatible and not safe_resume_pair: isolated_root=GATE_ROOT_BASE/f"{GATE_ROOT_BASE.name}_fresh_{time.time_ns()}"
selection=choose_full8_gate_run(GATE_ROOT,latest_path=latest,best_path=best,manifest_path=completed,latest_compatible=latest_compatible,best_compatible=best_compatible,completed_compatible=completed_compatible,isolated_output_dir=isolated_root)
if selection.action=="fresh_isolated":
    previous_root=GATE_ROOT; GATE_ROOT=selection.output_dir; GATE_TEMPLATE["output_dir"]=str(GATE_ROOT); write_gate_config()
    latest,best,completed=GATE_ROOT/"latest_gate.pt",GATE_ROOT/"best_gate.pt",GATE_ROOT/"run_manifest.json"
    resume_checkpoint=None
    print("Incompatible Full8 Gate artifacts preserved at",previous_root,"; starting fresh Full8 Gate training at",GATE_ROOT)
else: resume_checkpoint=selection.resume_checkpoint
print("Full8 Gate action:",selection.action,"output:",GATE_ROOT,"resume:",resume_checkpoint)
if SKIP_SCORE_REVIEW_EXPERIMENTAL and not synthetic_provenance:
    print("\\n*** EXPERIMENTAL: HUMAN SCORE REVIEW SKIPPED; train-only Score Audit remains active. No human review is claimed. Research-only. ***\\n")
if RUN_GATE_TRAIN and selection.action!="reuse_completed":
    gate_env=gpu_env(0)
    if SKIP_SCORE_REVIEW_EXPERIMENTAL: gate_env["MOESCORE_EXPERIMENTAL_SKIP_SCORE_REVIEW"]="1"
    else: gate_env.pop("MOESCORE_EXPERIMENTAL_SKIP_SCORE_REVIEW",None)
    command=[sys.executable,REPO_ROOT/"scripts/train_moe_gating.py","--config",GATE_CONFIG,"--report-json",LOG_ROOT/"full8_gate_training.json"]
    command+=gate_resume_args(selection)
    run_logged("full8_gate_train",command,env=gate_env)
elif selection.action=="reuse_completed": print("Reusing compatible completed Full8 run; no retraining.")
else: print("RUN_GATE_TRAIN=False; requiring a compatible existing best checkpoint.")
if not best.is_file() or not compatible_gate_checkpoint(best): raise FileNotFoundError("Compatible Full8 best_gate.pt required before evaluation and inference")
if selection.action in {"fresh","fresh_isolated"} and resume_checkpoint is not None: raise AssertionError("Fresh output directory retained a stale resume path")


## 11 — Official validation evaluation

Validation runs without optimizer updates. If train/validation WAV hashes overlap, the report must include the potentially optimistic-bias warning.


In [ ]:
if RUN_VALIDATION:
    VALIDATION_REPORT=LOG_ROOT/"full8_validation.json"
    run_logged("full8_validation",[sys.executable,REPO_ROOT/"scripts/evaluate_moe_gating.py","--config",GATE_CONFIG,"--checkpoint",best,"--device","cuda:0","--report-json",VALIDATION_REPORT])
    validation_report=json.loads(VALIDATION_REPORT.read_text())
    if validation_report.get("status")!="PASS" or validation_report.get("optimizer_steps")!=0: raise ValueError("Full8 validation failed or used optimizer updates")
    if EXPECTED_OVERLAP_DIAGNOSTIC.get("overlapping_sha256_count") and not validation_report.get("validation_metrics_warning"): raise ValueError("Overlapping validation metrics must be marked potentially biased")


## 12 — Label-free test inference, freeze predictions, then post-inference metrics

The test labels are opened only in this stage, after a successful label-free inference and SHA-256 freeze. Official labels use `wav_file_name,text,average_score`.


In [ ]:
SUBMISSION=GATE_ROOT/"submission.csv"; FROZEN_SUBMISSION=GATE_ROOT/"submission_frozen.csv"
TEST_FREEZE_MANIFEST=LOG_ROOT/"full8_test_prediction_freeze.json"; test_inference_completed=False
if RUN_TEST_INFERENCE:
    TEST_INFERENCE_REPORT=LOG_ROOT/"full8_test_inference.json"
    run_logged("full8_test_inference",[sys.executable,REPO_ROOT/"scripts/infer_moe_gating.py","--config",GATE_CONFIG,"--checkpoint",best,"--output",SUBMISSION,"--device","cuda:0","--report-json",TEST_INFERENCE_REPORT],env=gpu_env(0))
    inference_report=json.loads(TEST_INFERENCE_REPORT.read_text())
    if inference_report.get("status")!="PASS" or inference_report.get("labels_required") is not False or inference_report.get("sample_count")!=EXPECTED_COUNTS["test"]: raise ValueError("Full8 label-free inference failed or did not cover all test rows")
    with SUBMISSION.open(encoding="utf-8",newline="") as stream:
        reader=csv.DictReader(stream)
        if reader.fieldnames!=["wave_file_name","pred_score"]: raise ValueError(f"Unexpected submission schema: {reader.fieldnames}")
        predictions=list(reader)
    if len(predictions)!=EXPECTED_COUNTS["test"]: raise ValueError("Submission row count mismatch")
    prediction_sha=sha256_file(SUBMISSION); shutil.copy2(SUBMISSION,FROZEN_SUBMISSION); FROZEN_SUBMISSION.chmod(0o444)
    if sha256_file(FROZEN_SUBMISSION)!=prediction_sha: raise ValueError("Frozen prediction digest mismatch")
    TEST_FREEZE_MANIFEST.write_text(json.dumps({"status":"FROZEN_AFTER_LABEL_FREE_INFERENCE","sample_count":EXPECTED_COUNTS["test"],"prediction_file":str(FROZEN_SUBMISSION),"prediction_sha256":prediction_sha,"gate_checkpoint_sha256":sha256_file(best)},indent=2,sort_keys=True)+"\\n")
    test_inference_completed=True; print("Frozen label-free test predictions:",FROZEN_SUBMISSION,prediction_sha)
else: print("Test inference skipped; test labels will not be read.")


In [ ]:
TEST_METRICS_PATH=LOG_ROOT/"test_metrics.json"
if not RUN_TEST_METRICS:
    test_metrics_report={"status":"SKIPPED","run_purpose":"research_only","reason":"RUN_TEST_METRICS=False; test labels were not accessed."}; TEST_METRICS_PATH.write_text(json.dumps(test_metrics_report,indent=2)+"\\n")
elif not test_inference_completed:
    test_metrics_report={"status":"SKIPPED","run_purpose":"research_only","reason":"No inference/frozen predictions in this run; test labels were not accessed."}; TEST_METRICS_PATH.write_text(json.dumps(test_metrics_report,indent=2)+"\\n")
elif not ALLOW_POST_INFERENCE_TEST_LABEL_ACCESS:
    test_metrics_report={"status":"SKIPPED_LABEL_ACCESS_DISABLED","run_purpose":"research_only","reason":"Post-inference test-label access is disabled."}; TEST_METRICS_PATH.write_text(json.dumps(test_metrics_report,indent=2)+"\\n")
else:
    # First test-label read in this notebook is after frozen predictions.
    TEST_LABELS=DATA_ROOT/"meta_data"/"test_average.csv"
    if not TEST_LABELS.is_file():
        test_metrics_report={"status":"SKIPPED_LABELS_UNAVAILABLE","run_purpose":"research_only","reason":"Official labels unavailable; no metrics fabricated."}; TEST_METRICS_PATH.write_text(json.dumps(test_metrics_report,indent=2)+"\\n")
    else:
        frozen=json.loads(TEST_FREEZE_MANIFEST.read_text())
        if sha256_file(FROZEN_SUBMISSION)!=frozen["prediction_sha256"]: raise ValueError("Frozen predictions changed before metrics")
        from scripts.full7_test_metrics import evaluate_post_inference_metrics
        test_metrics_report=evaluate_post_inference_metrics(FROZEN_SUBMISSION,TEST_LABELS,TEST_METRICS_PATH,expected_count=EXPECTED_COUNTS["test"])
        test_metrics_report["submission_file"]=str(SUBMISSION); test_metrics_report["submission_sha256"]=sha256_file(SUBMISSION)
        if test_metrics_report["prediction_sha256"]!=test_metrics_report["submission_sha256"]: raise ValueError("Metrics were not calculated from the final submission")
        TEST_METRICS_PATH.write_text(json.dumps(test_metrics_report,indent=2,sort_keys=True)+"\\n")
print("Post-inference metrics:",test_metrics_report)


## 13 — Save checkpoints, reports and backup artifacts


In [ ]:
if CREATE_LOCAL_BACKUP:
    archive=WORK_ROOT/"moescore_full8_artifacts.tar.gz"
    artifacts=[(best,"gate/best_gate.pt"),(latest,"gate/latest_gate.pt"),(completed,"gate/run_manifest.json"),(SUBMISSION,"submission.csv"),(FROZEN_SUBMISSION,"submission_frozen.csv"),(TEST_FREEZE_MANIFEST,"test_prediction_freeze.json"),(GATE_CONFIG,"configs/moe_full8.yaml"),(E8_CONFIG,"configs/expert8_extraction.yaml"),(E8_EXTRACTION_REPORT,"reports/expert8_extraction.json"),(FULL8_BUILD_REPORT,"reports/full8_build.json"),(LOG_ROOT/"full8_score_audit.json","reports/full8_score_audit.json"),(LOG_ROOT/"full8_validation.json","reports/full8_validation.json"),(LOG_ROOT/"full8_gate_training.json","reports/full8_gate_training.json"),(TEST_METRICS_PATH,"reports/test_metrics.json"),(LOG_ROOT/"full8_test_inference.json","reports/test_inference.json")]
    with tarfile.open(archive,"w:gz") as tar:
        for artifact,member in artifacts:
            if Path(artifact).is_file(): tar.add(artifact,arcname=member,recursive=False)
        for log_path in sorted(LOG_ROOT.glob("*.log")): tar.add(log_path,arcname=str(Path("logs")/log_path.name),recursive=False)
    print("Full8 archive:",archive,"bytes:",archive.stat().st_size)
print("Full8 store/Gate/submission:",FULL8_ROOT,GATE_ROOT,SUBMISSION)
